# Bhajan Aabha — Autonomous Cloud Worker

Remote GPU production prototype. The user's computer is never used. This version performs one quota-light end-to-end generation: select a devotional opportunity, generate original Hindi lyrics/music, generate a short original devotional animation, assemble and QA an MP4. Publishing is intentionally gated until channel credentials are configured.

Copyright rule: never download, remix, pitch-shift, speed-change, or otherwise alter a copyrighted recording to evade detection. Use original generations, public-domain material, or explicitly licensed assets only.


In [ ]:
import os,sys,json,subprocess,gc
from pathlib import Path
import torch
WORK=Path('/kaggle/working'); OUT=WORK/'output'; OUT.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'No Kaggle GPU available; refusing any local/paid fallback.'
print('GPU:',torch.cuda.get_device_name(0)); print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
subprocess.run(['ffmpeg','-version'],stdout=subprocess.DEVNULL,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','git+https://github.com/huggingface/diffusers.git','transformers','accelerate','safetensors','sentencepiece','soundfile','pillow'],check=True)


In [ ]:
# Quota-light first run. Live trend discovery will replace this seed selector in the next stage.
opportunities=[
{'topic':'हनुमान जी','slug':'hanuman','title':'संकट हरने वाले हनुमान','visual':'Lord Hanuman in a serene Indian temple at sunrise, divine golden light, flowing flags, gentle devotional atmosphere'},
{'topic':'श्री कृष्ण','slug':'krishna','title':'श्याम तेरी बंसी','visual':'Lord Krishna beside the Yamuna at dawn, peacock feather, soft golden light, devotional atmosphere'},
{'topic':'महादेव','slug':'mahadev','title':'भोलेनाथ की शरण','visual':'Lord Shiva in a peaceful Himalayan temple setting, moonlight and sacred glow, devotional atmosphere'},
{'topic':'श्री राम','slug':'ram','title':'राम नाम की ज्योति','visual':'Lord Rama in a serene Ayodhya-inspired temple courtyard at sunrise, glowing lamps, devotional atmosphere'}]
choice=opportunities[0]
lyrics=f'''[verse]
जय जय {choice['topic']}, मन में तेरी ज्योति जले
तेरा नाम जो ले ले, उसके दुख सब दूर चले
शरण तेरी पावन है, चरणों में संसार मिले

[chorus]
जय जय {choice['topic']}, मन में तेरी ज्योति जले
भक्ति की इस राह में, हर मन तेरा नाम कहे

[verse]
भोर हुई तो नाम तेरा, सांसों में संगीत बने
अंधियारे में दीप जलाए, मन को नई उम्मीद मिले
तेरी कृपा की छांव मिले तो, हर राह सरल बने

[chorus]
जय जय {choice['topic']}, मन में तेरी ज्योति जले'''
print(json.dumps(choice,ensure_ascii=False)); print(lyrics)


In [ ]:
# ACE-Step 1.5 Turbo: original music with supplied lyrics. MIT-licensed model; no source recording is used.
import soundfile as sf
from diffusers import AceStepPipeline
music_pipe=AceStepPipeline.from_pretrained('ACE-Step/acestep-v15-xl-turbo-diffusers',torch_dtype=torch.bfloat16)
music_pipe=music_pipe.to('cuda'); music_pipe.vae.enable_tiling()
music=music_pipe(prompt=f'Hindi devotional bhajan about {choice["topic"]}, warm expressive lead vocal, harmonium, bansuri flute, tabla and tanpura, uplifting spiritual mood, original melody',lyrics=lyrics,audio_duration=10.0,vocal_language='hi',num_inference_steps=8,generator=torch.Generator(device='cuda').manual_seed(42)).audios[0]
music_path=OUT/'original_bhajan.wav'; sf.write(music_path,music.T.cpu().float().numpy(),music_pipe.sample_rate)
print('Music saved:',music_path)
del music_pipe,music; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# CogVideoX-2B: Apache-2.0 short devotional animation. It is generated from text; no copyrighted footage is downloaded.
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
video_pipe=CogVideoXPipeline.from_pretrained('zai-org/CogVideoX-2b',torch_dtype=torch.float16)
video_pipe.enable_model_cpu_offload()
frames=video_pipe(prompt=f'Reverent cinematic devotional scene: {choice["visual"]}, subtle natural movement, gentle lamp and cloth motion, slow camera push-in, richly detailed Indian devotional art, no text, no watermark',num_frames=49,guidance_scale=6,num_inference_steps=20).frames[0]
visual_path=OUT/'devotional_clip.mp4'; export_to_video(frames,str(visual_path),fps=8)
print('Animation saved:',visual_path)
del video_pipe,frames; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# Assemble a vertical 10-second video, add the exact generated lyrics, branding and QA.
music_path=str(music_path); visual_path=str(visual_path); final_path=OUT/f'Bhajan_Aabha_{choice["slug"]}_prototype.mp4'; srt_path=OUT/'lyrics.srt'
lines=[x.strip() for x in lyrics.splitlines() if x.strip() and not x.startswith('[')]
def st(t):
 h=int(t//3600);m=int((t%3600)//60);s=int(t%60);ms=int((t-int(t))*1000);return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'
with open(srt_path,'w',encoding='utf-8') as f:
 n=len(lines); step=10/max(1,n)
 for i,line in enumerate(lines): f.write(f'{i+1}\n{st(i*step)} --> {st(min(10,(i+1)*step))}\n{line}\n\n')
vf=f"scale=1080:1920:force_original_aspect_ratio=decrease,pad=1080:1920:(ow-iw)/2:(oh-ih)/2,subtitles='{srt_path}':force_style='FontName=Noto Sans Devanagari,FontSize=18,Alignment=2,MarginV=120,Outline=2'"
subprocess.run(['ffmpeg','-y','-stream_loop','-1','-i',visual_path,'-i',music_path,'-t','10','-vf',vf,'-c:v','libx264','-preset','veryfast','-crf','21','-c:a','aac','-b:a','192k','-movflags','+faststart',str(final_path)],check=True)
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1',str(final_path)],capture_output=True,text=True,check=True)
streams=subprocess.run(['ffprobe','-v','error','-show_entries','stream=codec_type','-of','csv=p=0',str(final_path)],capture_output=True,text=True,check=True).stdout
assert final_path.exists() and final_path.stat().st_size>500000 and float(probe.stdout.strip())>5 and 'video' in streams and 'audio' in streams
state={'channel':'Bhajan Aabha','topic':choice['topic'],'output':str(final_path),'duration_sec':round(float(probe.stdout.strip()),2),'gpu':torch.cuda.get_device_name(0),'copyright_mode':'original_generation_only','publish_status':'PENDING_CHANNEL_AUTH'}
(OUT/'run_state.json').write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(state,ensure_ascii=False,indent=2))
